# USMA1Q — Méthodes Numériques
## Exercices Séance 4 — Tableaux en mémoire, matrices & précision numérique — SOLUTIONS

## Partie 1 — Vues et copies

### Exercice 1.1 — Tableau d'éprouvettes

On dispose de données pour 10 éprouvettes d'acier. Chaque ligne correspond à une éprouvette, les colonnes sont :  
- colonne 0 : résistance à la traction (MPa)  
- colonne 1 : allongement à la rupture (%)  
- colonne 2 : dureté (HV)

1. Créez le tableau `data` à partir des valeurs ci-dessous.  
2. Extrayez la colonne de dureté dans une variable `durete`. Vérifiez avec `np.shares_memory()` que c'est bien une vue.  
3. Modifiez `durete[0] = 0`. Observez comment `data` a changé.  
4. Repartez du tableau original (recréez-le). Extrayez la colonne de dureté avec `.copy()`. Refaites la modification et vérifiez que `data` est intact.  
5. À l'aide de `%timeit`, comparez la vitesse de `np.sum(big, axis=0)` vs `np.sum(big, axis=1)` pour un grand tableau `big = np.random.rand(10000, 10000)`. Quel axe est plus rapide ? Pourquoi ?

In [ ]:
import numpy as np

# Données brutes : [UTS (MPa), allongement (%), dureté (HV)]
valeurs = [
    [510, 22, 152],
    [495, 24, 148],
    [522, 21, 156],
    [503, 23, 150],
    [515, 22, 154],
    [498, 25, 147],
    [519, 21, 155],
    [507, 23, 151],
    [501, 24, 149],
    [513, 22, 153],
]

# 1. Créer le tableau
data = np.array(valeurs)
print("data.shape :", data.shape)
print("data.dtype :", data.dtype)

In [ ]:
# 2. Extraire la colonne de dureté — est-ce une vue ?
durete = data[:, 2]
print("Partage mémoire data/durete :", np.shares_memory(data, durete))

# 3. Modifier durete[0] — observer l'effet sur data
durete[0] = 0
print("Après durete[0] = 0, data[0] :", data[0])
# data[0, 2] a aussi changé — durete est une vue, pas une copie

# 4. Repartir du tableau original, extraire la colonne avec .copy()
data = np.array(valeurs)   # recréer le tableau original
durete_copie = data[:, 2].copy()   # copie indépendante
durete_copie[0] = 0
print("data[0] après modification de la copie :", data[0])
# data est intact — la copie est indépendante

In [ ]:
# 5. Benchmark axis=0 vs axis=1
# NB : ceci peut prendre quelques secondes
big = np.random.rand(5000, 5000)

import time

t0 = time.perf_counter()
for _ in range(10):
    np.sum(big, axis=0)
t1 = time.perf_counter()
print("axis=0 (10 répétitions) :", round(t1 - t0, 3), "s")

t0 = time.perf_counter()
for _ in range(10):
    np.sum(big, axis=1)
t1 = time.perf_counter()
print("axis=1 (10 répétitions) :", round(t1 - t0, 3), "s")

# Explication : NumPy utilise l'ordre C (ligne-majeur) — les éléments d'une même ligne sont contigus en mémoire.
# La somme axis=1 (le long des colonnes d'une même ligne) parcourt donc des données contiguës → plus favorable 
# pour l'ordinateur. La somme axis=0 saute d'une ligne à l'autre, ce qui est moins favorable. En pratique, axis=1 est souvent légèrement plus rapide.

## Partie 2 — Opérations matricielles

### Exercice 2.1 — Système de 3 ressorts en série

Trois ressorts de raideurs $k_1 = 100$ N/mm, $k_2 = 150$ N/mm, $k_3 = 80$ N/mm sont assemblés en série. En éléments finis 1D, la matrice de rigidité assemblée pour les degrés de liberté internes $(u_1, u_2, u_3)$ est :

$$K = \begin{pmatrix} k_1 & -k_1 & 0 \\ -k_1 & k_1+k_2 & -k_2 \\ 0 & -k_2 & k_2+k_3 \end{pmatrix}$$

1. Construisez `K` à la main avec `np.array()`.  
2. Construisez le même `K` de façon programmatique en utilisant `np.diag()` — en une ou deux lignes.  
3. Vérifiez que les deux matrices sont égales avec `np.allclose()`.  
4. Calculez le vecteur de déplacement $u$ si la force appliquée est $F = [0, 0, 500]$ N (les deux premiers nœuds sont libres, le troisième est soumis à la force) via `np.linalg.solve(K, F)` (*on approfondit cela en séance 5*).  
5. Calculez la force résultante `K @ u` et vérifiez qu'elle est bien égale à `F`.

In [ ]:
import numpy as np

k1, k2, k3 = 100, 150, 80

# 1. Construction directe
K_direct = np.array([
    [ k1,        -k1,        0       ],
    [-k1,         k1 + k2,  -k2      ],
    [  0,        -k2,        k2 + k3 ],
], dtype=float)
print("K =", K_direct)

In [ ]:
# 2. Construction programmatique avec np.diag()
# Diagonale principale   : [k1, k1+k2, k2+k3]
# Sur-diagonale  (k=+1)  : [-k1, -k2]
# Sous-diagonale (k=-1)  : [-k1, -k2]
K_prog = np.diag([k1,  k1 + k2,  k2 + k3], k=0) + np.diag([-k1, -k2], k=1) + np.diag([-k1, -k2], k=-1)

# 3. Vérification
print("Matrices égales :", np.allclose(K_direct, K_prog))

In [ ]:
# 4. Résolution Ku = F
F = np.array([0.0, 0.0, 500.0])

u = np.linalg.solve(K_direct, F)
print("Vecteur déplacement u (mm) :", np.round(u, 4))
# Ils sont tous le meme valeur parce qu'on n'a pas fixé le déplacement à un noeud donc c'est un système indéterminé, mais np.linalg.solve trouve une solution particulière (la translation horizontale de tout le système) qui satisfait l'équation Ku=F.

# 5. Vérification : K @ u doit être égal à F
F_recalc = K_direct @ u
print("K @ u               :", np.round(F_recalc, 6))
print("K @ u == F (allclose):", np.allclose(F_recalc, F))

### Exercice 2.2 — Opérations sur une matrice de mesures

Vous avez mesuré les modules de Young de 4 matériaux selon 3 directions différentes (résultats en GPa) :

|           | Direction 1 | Direction 2 | Direction 3 |
|-----------|-------------|-------------|-------------|
| Matériau A | 210 | 208 | 212 |
| Matériau B | 195 | 198 | 193 |
| Matériau C | 70  | 72  | 69  |
| Matériau D | 116 | 114 | 118 |

**Rappel — norme de Frobenius :** pour une matrice $M$ de taille $m \times n$, la norme de Frobenius est la racine de la somme des carrés de tous les éléments :
$$\|M\|_F = \sqrt{\sum_{i=1}^{m}\sum_{j=1}^{n} m_{ij}^2}$$
C'est l'extension naturelle de la norme euclidienne aux matrices. Dans NumPy : `np.linalg.norm(M)` (c'est la norme par défaut pour les matrices 2D).

1. Créez la matrice `E` (4 × 3).  
2. Calculez la moyenne par matériau (axis=1) et par direction (axis=0).  
3. Calculez la matrice de covariance entre matériaux via `np.cov(E)` (*chaque ligne est une variable, chaque colonne est une observation*).  
4. Calculez la norme de Frobenius de `E`.  
5. Calculez le produit $E^T E$ (matrice 3×3). Quelle est sa trace ? Comparez à la norme de Frobenius au carré.

In [ ]:
import numpy as np

E = np.array([
    [210, 208, 212],
    [195, 198, 193],
    [ 70,  72,  69],
    [116, 114, 118],
], dtype=float)

# 2. Moyennes
moy_materiaux  = np.mean(E, axis=1)   # moyenne sur les colonnes (directions)
moy_directions = np.mean(E, axis=0)   # moyenne sur les lignes (matériaux)
print("Moyenne par matériau :", moy_materiaux, "GPa")
print("Moyenne par direction :", moy_directions, "GPa")

# 3. Matrice de covariance (4×4 — une ligne par matériau, une colonne par direction)
cov_E = np.cov(E)
print("Matrice de covariance (4x4) :")
print(np.round(cov_E, 2))
# Chaque coefficient (i,j) mesure comment les modules des matériaux i et j varient ensemble selon les directions.
# Une valeur positive signifie qu'ils varient dans le même sens.

# 4. Norme de Frobenius
norm_F = np.linalg.norm(E)
print("Norme de Frobenius ||E||_F :", round(norm_F, 4), "GPa")

# 5. E^T @ E et sa trace
EtE = E.T @ E
print("E^T @ E :")
print(np.round(EtE, 2))
trace_EtE = np.trace(EtE)
print("trace(E^T @ E) :", round(trace_EtE, 4))
print("||E||_F²       :", round(norm_F**2, 4))
print("Egalite        :", np.isclose(trace_EtE, norm_F**2))
# Propriété générale : trace(E^T @ E) = ||E||_F² (c'est la somme des carrés de tous les éléments)

## Partie 3 — Précision et virgule flottante

### Exercice 3.1 — Epsilon machine et petites perturbations

1. Vérifiez que `0.1 + 0.2 == 0.3` renvoie `False`. Affichez `0.1 + 0.2` en pleine précision avec `repr()`.  
2. Affichez `np.finfo(np.float64).eps`. Quelle en est la signification ?  
3. Calculez `(1 + 1e-16) - 1`. Expliquez le résultat.  
4. Calculez `(1 + 1e-8) - 1`. Est-ce exact ? Quelle est l'erreur relative ?  
5. Calculez `(1 + 1e-8) - 1` en utilisant `np.float128`. La précision s'améliore-t-elle ?

In [ ]:
import numpy as np

# 1. 0.1 + 0.2
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("repr(0.1 + 0.2)  :", repr(0.1 + 0.2))

# 2. Epsilon machine
eps = np.finfo(np.float64).eps
print("np.finfo(np.float64).eps :", eps)
# L'epsilon machine est le plus petit δ tel que 1.0 + δ ≠ 1.0 en float64.

In [ ]:
# 3. (1 + 1e-16) - 1
resultat_16 = (1 + 1e-16) - 1
print("(1 + 1e-16) - 1 :", resultat_16)
# 1e-16 < eps/2 ≈ 1.11e-16 → 1.0 + 1e-16 est arrondi à 1.0 exactement.
# La soustraction donne 0.0 — perte totale de l'information.

# 4. (1 + 1e-8) - 1 et erreur relative
resultat_8   = (1 + 1e-8) - 1
valeur_exacte = 1e-8
erreur_rel_4 = abs(resultat_8 - valeur_exacte) / valeur_exacte
print("(1 + 1e-8) - 1    :", resultat_8)
print("Erreur relative   :", erreur_rel_4)
# 1e-8 >> eps/2, donc 1 + 1e-8 est représentable — mais avec une petite erreur d'arrondi de l'ordre de l'epsilon machine.

# 5. Comparaison avec np.float128 (si disponible)
x_128 = np.float128(1) + np.float128(1e-8) - np.float128(1)
erreur_rel_5 = abs(float(x_128) - valeur_exacte) / valeur_exacte
print("float128 (1 + 1e-8) - 1 :", x_128)
print("Erreur relative float128 :", erreur_rel_5)
# float128 offre ~18-19 chiffres significatifs contre ~15 pour float64 : l'erreur relative est significativement plus petite.

### Exercice 3.2 — Somme de la série $\sum_{n=1}^{N} \frac{1}{n^2}$

La série $\sum_{n=1}^{\infty} \frac{1}{n^2}$ converge vers $\pi^2/6$ (résultat de Bâle).

1. Pour $N = 10\,000$, calculez la somme **croissante** (de $n=1$ à $N$) et la somme **décroissante** (de $n=N$ à $1$). Comparez les deux résultats à $\pi^2/6$.  
2. Quelle méthode est plus précise ? Pourquoi ?  
3. *(Bonus)* Essayez avec `np.float32` et comparez les erreurs.

In [ ]:
import numpy as np

N = 10000
vrai = np.pi**2 / 6

# 1. Somme croissante (n=1 à N) et décroissante (n=N à 1)
n_croissant = np.arange(1, N + 1)
somme_croissante   = np.sum(1.0 / n_croissant**2)
somme_decroissante = np.sum(1.0 / n_croissant[::-1]**2)  # ordre inversé

print("Valeur vraie (pi²/6)     :", vrai)
print("Somme croissante (1->N)  :", somme_croissante,   "  erreur :", abs(somme_croissante - vrai))
print("Somme décroissante (N->1):", somme_decroissante, "  erreur :", abs(somme_decroissante - vrai))

# 2. En principe la somme décroissante (N→1) dû être plus précise, mais en réalité les manipulations internes de numpy peuvent 
# faire que les deux soient similaires ou même renverser nos attentes.

In [ ]:
# 3. Bonus : comparaison float32 vs float64
n_f32 = n_croissant.astype(np.float32)
somme_f32_croissante   = np.sum(np.float32(1.0) / n_f32**2)
somme_f32_decroissante = np.sum(np.float32(1.0) / n_f32[::-1]**2)

print("float32 croissante   :", somme_f32_croissante,   " erreur :", abs(float(somme_f32_croissante)  - vrai))
print("float32 décroissante :", somme_f32_decroissante, " erreur :", abs(float(somme_f32_decroissante) - vrai))
# Les erreurs float32 sont ~100-1000× plus grandes que celles de float64 :
# float32 ne dispose que d'environ 7 chiffres significatifs contre 15 pour float64.

## Partie 4 — Propagation d'incertitude

### Exercice 4.1 — Contrainte avec incertitude

Cinq éprouvettes cylindriques ont les mesures suivantes :

| Éprouvette | Force rupture (N) | Diamètre (mm) |
|------------|-------------------|----------------|
| 1 | 45 200 | 10.02 |
| 2 | 43 800 | 9.98  |
| 3 | 46 100 | 10.05 |
| 4 | 44 500 | 10.00 |
| 5 | 45 700 | 10.03 |

L'incertitude instrumentale est $\delta F = 100$ N et $\delta d = 0{,}02$ mm.

La contrainte d'ingénierie est $\sigma = F / A = 4F / (\pi d^2)$ et son incertitude :

$$\delta\sigma = \sigma \sqrt{\left(\frac{\delta F}{F}\right)^2 + 4\left(\frac{\delta d}{d}\right)^2}$$

1. **Sans boucle** (`for`), calculez vectoriellement $\sigma$ et $\delta\sigma$ pour les 5 éprouvettes.  
2. Affichez les résultats sous forme de tableau : `"σ = xxx.x ± y.y MPa"`.  
3. Quelle est la contrainte moyenne ? Quel est l'écart-type des contraintes mesurées ?  
4. Pour quelle éprouvette l'incertitude instrumentale est-elle relativement la plus grande ?

In [ ]:
import numpy as np

forces    = np.array([45200, 43800, 46100, 44500, 45700], dtype=float)  # N
diametres = np.array([10.02, 9.98, 10.05, 10.00, 10.03])               # mm

delta_F = 100.0   # N
delta_d = 0.02    # mm

# 1. Calcul vectoriel (sans boucle for) — formule : delta_sigma = sigma * sqrt((dF/F)² + 4*(dd/d)²)
aires       = np.pi * (diametres / 2) ** 2
sigma       = forces / aires
delta_sigma = sigma * np.sqrt((delta_F / forces)**2 + 4 * (delta_d / diametres)**2)

# 2. Affichage formaté (boucle de présentation uniquement, pas de calcul)
for i in range(len(forces)):
    print("Eprouvette ", i + 1, ": sigma = ", round(sigma[i], 1), " +/- ", round(delta_sigma[i], 2), " MPa")
    # J'ai arrondi à 2 décimales pour delta_sigma dans les solutions pour mieux refléter la variation, 
    # même si les exercices ont donné 1 décimale.

# 3. Contrainte moyenne et écart-type
print("Contrainte moyenne :", round(np.mean(sigma), 1), "MPa")
print("Ecart-type         :", round(np.std(sigma), 1), "MPa")

# 4. Éprouvette avec l'incertitude relative la plus grande
incert_rel = delta_sigma / sigma
idx_max = np.argmax(incert_rel)
print("Incertitude relative max : eprouvette", idx_max + 1, "(", round(incert_rel[idx_max] * 100, 4), "%)")
# L'éprouvettes avec la plus petite force tendent à avoir l'incertitude relative la plus grande.

### Exercice 4.2 — Propagation sur un volume

Une pièce cylindrique creuse a les dimensions nominales :  
- diamètre extérieur $D = 50$ mm ± 0,05 mm  
- diamètre intérieur $d = 40$ mm ± 0,05 mm  
- longueur $L = 200$ mm ± 0,1 mm

Le volume est $V = \frac{\pi}{4}(D^2 - d^2) L$.

1. Calculez le volume nominal.  
2. Dérivez analytiquement $\delta V / V$ en fonction de $\delta D$, $\delta d$, $\delta L$.  
3. Calculez $\delta V$ numériquement.  
4. **Bonus / vérification :** estimez $\delta V$ par différences finies : $V(D + \delta D, d, L) - V(D, d, L)$, etc.  
5. Quelle source d'incertitude (D, d ou L) contribue le plus à $\delta V$ ? Justifiez.

In [ ]:
import numpy as np

D, delta_D = 50.0, 0.05    # mm
d, delta_d = 40.0, 0.05    # mm
L, delta_L = 200.0, 0.1    # mm

# 1. Volume nominal
V = np.pi / 4 * (D**2 - d**2) * L
print("Volume nominal V =", round(V, 2), "mm^3")

# 2 & 3. Dérivées partielles analytiques et propagation
# dV/dD =  pi * D * L / 2
# dV/dd = -pi * d * L / 2
# dV/dL =  pi / 4 * (D² - d²)
dV_dD = np.pi * D * L / 2
dV_dd = -np.pi * d * L / 2
dV_dL = np.pi / 4 * (D**2 - d**2)

delta_V = np.sqrt((dV_dD * delta_D)**2 + (dV_dd * delta_d)**2 + (dV_dL * delta_L)**2)
print("delta_V =", round(delta_V, 2), "mm^3")
print("delta_V/V =", round(delta_V / V * 100, 4), "%")

# 4. Bonus : estimation par différences finies
dV_D_fd = np.pi / 4 * ((D + delta_D)**2 - d**2) * L - V
dV_d_fd = np.pi / 4 * (D**2 - (d + delta_d)**2) * L - V
dV_L_fd = np.pi / 4 * (D**2 - d**2) * (L + delta_L) - V
delta_V_fd = np.sqrt(dV_D_fd**2 + dV_d_fd**2 + dV_L_fd**2)
print("delta_V (differences finies) =", round(delta_V_fd, 2), "mm^3")

# 5. Contribution relative de chaque source d'incertitude
contrib_D = (dV_dD * delta_D)**2 / delta_V**2 * 100
contrib_d = (dV_dd * delta_d)**2 / delta_V**2 * 100
contrib_L = (dV_dL * delta_L)**2 / delta_V**2 * 100
print("Contribution D :", round(contrib_D, 1), "%")
print("Contribution d :", round(contrib_d, 1), "%")
print("Contribution L :", round(contrib_L, 1), "%")
# D contribue le plus (~61 %) car son rayon de levier (D=50 mm) est plus grand
# que celui de d (40 mm). L contribue très peu (<1 %) car delta_L/L est petit.